## EV Resale Price Regression

#### 1. Data Quality Audit
#### Perform a complete audit of the dataset:
#### • shape
#### • datatypes
#### • missing values
#### • duplicate records
#### • unique vehicle IDs
#### • categorical distributions
#### Identify every data-quality problem before modifying the dataset.

In [2]:
import pandas as pd

df = pd.read_csv("C:\\Users\\rohit\\Downloads\\EV_Resale_Price_Regression.csv")

print("Shape of Dataset:")
print(df.shape)

print("\nData Types:")
print(df.dtypes)

print("\nMissing Values:")
print(df.isnull().sum())

print("\nDuplicate Records:")
print(df.duplicated().sum())

print("\nUnique Vehicle IDs:")
print(df["vehicle_id"].nunique())

categorical_cols = df.select_dtypes(include="object").columns

print("\nCategorical Distributions:")
for col in categorical_cols:
    print(f"\n{col}")
    print(df[col].value_counts())

Shape of Dataset:
(1000, 15)

Data Types:
vehicle_id               object
listing_date             object
manufacture_year          int64
brand                    object
vehicle_type             object
battery_capacity_kwh    float64
battery_health_pct      float64
range_km                float64
km_driven               float64
charging_time_hr        float64
fast_charging            object
owner_count               int64
city                     object
service_history          object
resale_price            float64
dtype: object

Missing Values:
vehicle_id               0
listing_date             0
manufacture_year         0
brand                    0
vehicle_type             0
battery_capacity_kwh    25
battery_health_pct      30
range_km                20
km_driven                0
charging_time_hr        24
fast_charging            0
owner_count              0
city                     0
service_history         30
resale_price             0
dtype: int64

Duplicate Records:
0

Unique

#### 2. Duplicate Vehicle Investigation
#### vehicle_id is expected to uniquely identify a vehicle.
#### Determine:
#### • how many duplicated vehicle_id values exist
#### • which vehicle IDs are duplicated
#### • how many records are affected
#### Then decide how you would handle these records without blindly using drop_duplicates().

In [3]:
duplicate_mask = df["vehicle_id"].duplicated(keep=False)

print("Duplicated vehicle IDs:", df.loc[duplicate_mask, "vehicle_id"].nunique())

print("Duplicated vehicle IDs:", df.loc[duplicate_mask, "vehicle_id"].nunique())

print("\nDuplicate Vehicle IDs:")
print(df.loc[duplicate_mask, "vehicle_id"].unique())

print("\nTotal affected records:", duplicate_mask.sum())

print("\nDuplicate Records:")
print(df[duplicate_mask].sort_values("vehicle_id"))

Duplicated vehicle IDs: 6
Duplicated vehicle IDs: 6

Duplicate Vehicle IDs:
['EV-20515' 'EV-20653' 'EV-20011' 'EV-20544' 'EV-20193' 'EV-20279']

Total affected records: 12

Duplicate Records:
    vehicle_id listing_date  manufacture_year       brand vehicle_type  \
119   EV-20011   2025-04-30              2016      Nexora        Sedan   
852   EV-20011   2027-05-03              2016      Nexora        Sedan   
655   EV-20193   2026-10-18              2023     Atheron    Crossover   
797   EV-20193   2027-03-09              2023     Atheron    Crossover   
816   EV-20279   2027-03-28              2019    E-Motion          SUV   
857   EV-20279   2027-05-08              2019    E-Motion          SUV   
72    EV-20515   2025-03-14              2016  GreenDrive    Hatchback   
621   EV-20515   2026-09-14              2016  GreenDrive    Hatchback   
133   EV-20544   2025-05-14              2024    E-Motion        Sedan   
280   EV-20544   2025-10-08              2024    E-Motion        Sed

#### 3. Date Conversion & Validation
##### Convert listing_date into a proper datetime column.
##### Then investigate:
##### • earliest listing date
##### • latest listing date
##### • invalid/missing dates
##### • whether the date column is suitable for feature engineering.

In [4]:
df["listing_date"] = pd.to_datetime(df["listing_date"], errors="coerce")

print("Earliest Listing Date:")
print(df["listing_date"].min())


print("\nLatest Listing Date:")
print(df["listing_date"].max())


print("\nInvalid/Missing Dates:")
print(df["listing_date"].isnull().sum())

print("\nSuitable for Feature Engineering:",
      df["listing_date"].notnull().all())

Earliest Listing Date:
2025-01-01 00:00:00

Latest Listing Date:
2027-09-27 00:00:00

Invalid/Missing Dates:
0

Suitable for Feature Engineering: True


#### 4. Vehicle Age Feature
##### Create vehicle_age using:
##### listing year − manufacture year
##### Then identify whether any vehicle has:
##### • age < 0
##### • age = 0
##### • unusually high age

In [5]:
df["listing_date"] = pd.to_datetime(df["listing_date"], errors="coerce")

df["vehicle_age"] = df["listing_date"].dt.year - df["manufacture_year"]


print("Vehicles with age < 0:")
print(df[df["vehicle_age"] < 0])


print("\nVehicles with age = 0:")
print(df[df["vehicle_age"] == 0])

print("\nVehicles with unusually high age:")
print(df[df["vehicle_age"] > 20])

print("\nVehicle Age Summary:")
print(df["vehicle_age"].describe())

Vehicles with age < 0:
Empty DataFrame
Columns: [vehicle_id, listing_date, manufacture_year, brand, vehicle_type, battery_capacity_kwh, battery_health_pct, range_km, km_driven, charging_time_hr, fast_charging, owner_count, city, service_history, resale_price, vehicle_age]
Index: []

Vehicles with age = 0:
    vehicle_id listing_date  manufacture_year       brand vehicle_type  \
1     EV-20530   2025-01-02              2025      Nexora    Crossover   
8     EV-20708   2025-01-09              2025      Nexora        Sedan   
9     EV-20059   2025-01-10              2025     Atheron        Sedan   
12    EV-20381   2025-01-13              2025      Nexora        Sedan   
19    EV-20301   2025-01-20              2025    E-Motion        Sedan   
22    EV-20842   2025-01-23              2025     Atheron    Crossover   
31    EV-20891   2025-02-01              2025      Voltix    Hatchback   
65    EV-20356   2025-03-07              2025     Atheron    Crossover   
91    EV-20607   2025-04-02

#### 5. Battery Data Imputation
##### The following columns contain missing values:
##### battery_capacity_kwh, battery_health_pct, range_km, charging_time_hr
##### Develop an appropriate missing-value strategy for each column.
##### Do not automatically use the same statistic for every column.

##### Explain why you selected mean, median, or another strategy.

In [7]:
df["battery_capacity_kwh"] = df["battery_capacity_kwh"].fillna(df["battery_capacity_kwh"].median())

df["battery_health_pct"] = df["battery_health_pct"].fillna(df["battery_health_pct"].mean())

df["range_km"] = df["range_km"].fillna(df["range_km"].median())

df["charging_time_hr"] = df["charging_time_hr"].fillna(df["charging_time_hr"].median())

df = df.fillna({
    "battery_capacity_kwh": df["battery_capacity_kwh"].median(),
    "battery_health_pct": df["battery_health_pct"].mean(),
    "range_km": df["range_km"].median(),
    "charging_time_hr": df["charging_time_hr"].median()
})

print(df[[
    "battery_capacity_kwh",
    "battery_health_pct",
    "range_km",
    "charging_time_hr"
]].isnull().sum())

battery_capacity_kwh    0
battery_health_pct      0
range_km                0
charging_time_hr        0
dtype: int64
